# Porkchop: Lambert arcs on a grid

Uses the Python mirror `pymiss` (same formulas and names as `src/physics`). Run from the repo root.


In [ ]:
import sys, math
sys.path.insert(0, '..' if 'notebooks' in __import__('os').getcwd() else '.')
from pymiss.physics.constants import AU, MU_SUN, DAY
from pymiss.physics.kepler import Elements, propagate
from pymiss.physics.lambert import lambert
from pymiss.physics.estimates import c3_from_vinf, payload_for_c3, launcher_tier
# Same scenario as the lab defaults: HOME circular at 1 AU, target a = 1.4 AU, e = 0.25.
home = Elements(AU, 0.0, 0.0, 0.0, 0.0)
target = Elements(1.4 * AU, 0.25, 1.0, 2.0, 0.0)
tier = launcher_tier(2)
best = None
for launch in range(0, 400, 10):
    hx, hy, hvx, hvy = propagate(home, MU_SUN, launch * DAY)
    for flight in range(30, 420, 10):
        tx, ty, _, _ = propagate(target, MU_SUN, (launch + flight) * DAY)
        sol = lambert(hx, hy, tx, ty, flight * DAY, MU_SUN)
        if sol is None:
            continue
        c3 = c3_from_vinf(math.hypot(sol.v1x - hvx, sol.v1y - hvy))
        if best is None or c3 < best[0]:
            best = (c3, launch, flight, payload_for_c3(tier, c3))
print('min C3 %.2f km2/s2 at launch day %d, flight %d d, payload %.0f kg' % best)